# 🍇 GRAPE CNN — MASTER MODEL TRAINING
### Project: Plant Disease Detection (Computer Vision)
**Architecture:** MobileNetV2 (Transfer Learning) + Data Augmentation  
**Dataset:** `3rd Preprocessing/grape_processed_data.npz` (Internal PlantVillage + External Natural)  
**Target Classes (3):** `Healthy (0)`, `Black_Rot (1)`, `Leaf_Blight (2)`  
**Protocol:** Standardized transfer learning pipeline with EarlyStopping and ReduceLROnPlateau.



In [ ]:
# ================================================================
# 🍇 GRAPE — FINAL MODEL TRAINING PRE-CHECK
# READ-ONLY AUDIT
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np

# ------------------------------------------------
# PROJECT PATH
# ------------------------------------------------
BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
if not os.path.exists(BASE):
    BASE = r"G:\My Drive\Plant Disease Detection (Computer Vision)"

PREPROCESS = os.path.join(BASE, "3rd Preprocessing")
TRAINING   = os.path.join(BASE, "4th Model_Training")
MODEL_DIR  = os.path.join(BASE, "6th Trained_Model")

NPZ_PATH   = os.path.join(PREPROCESS, "grape_processed_data.npz")

print("=" * 75)
print("🍇 GRAPE DATASET & ENVIRONMENT AUDIT")
print("=" * 75)
print("NPZ Path:", NPZ_PATH)
if os.path.exists(NPZ_PATH):
    data = np.load(NPZ_PATH)
    print("✅ NPZ File Found! Size:", round(os.path.getsize(NPZ_PATH)/(1024**2), 2), "MB")
    print("Arrays in NPZ:", data.files)
    print("X shape:", data['X'].shape, "dtype:", data['X'].dtype)
    print("y shape:", data['y'].shape, "dtype:", data['y'].dtype)
    print("Classes:", data['class_names'])
else:
    print("❌ NPZ File Not Found! Please run Grape_Preprocessing.ipynb first.")



Mounted at /content/drive
🍇 GRAPE DATASET & ENVIRONMENT AUDIT
NPZ Path: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/grape_processed_data.npz
✅ NPZ File Found! Size: 183.24 MB
Arrays in NPZ: ['X', 'y', 'class_names', 'source']
X shape: (1800, 224, 224, 3) dtype: uint8
y shape: (1800,) dtype: int64
Classes: ['Healthy' 'Black_Rot' 'Leaf_Blight']


In [ ]:
# ================================================================
# 🍇 GRAPE CNN — FINAL MASTER TRAINING PIPELINE
# Transfer Learning via MobileNetV2 + Data Augmentation
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import csv
import random
import hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split

# ================================================================
# 1. REPRODUCIBILITY SEED
# ================================================================

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ================================================================
# 2. PATHS & ARTIFACTS
# ================================================================

BASE = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
if not os.path.exists(BASE):
    BASE = r"G:\My Drive\Plant Disease Detection (Computer Vision)"

NPZ_PATH   = os.path.join(BASE, "3rd Preprocessing", "grape_processed_data.npz")
MODEL_DIR  = os.path.join(BASE, "6th Trained_Model")
TRAIN_DIR  = os.path.join(BASE, "4th Model_Training")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(MODEL_DIR, "grape_cnn_best.keras")
LOG_CSV_PATH    = os.path.join(TRAIN_DIR, "grape_CNN_training_log.csv")
SUMMARY_JSON    = os.path.join(TRAIN_DIR, "grape_CNN_training_summary.json")

# ================================================================
# 3. LOAD DATASET
# ================================================================

print("=" * 75)
print("🍇 1. LOADING GRAPE DATASET")
print("=" * 75)

data = np.load(NPZ_PATH)
X = data['X']
y = data['y']
class_names = data['class_names']
source = data['source']

print("Total samples :", len(X))
print("X shape       :", X.shape, f", dtype: {X.dtype}")
print("y shape       :", y.shape, f", dtype: {y.dtype}")
print("Classes       :", class_names.tolist())

# ================================================================
# 4. STRATIFIED TRAIN / VAL / TEST SPLIT (70% / 15% / 15%)
# ================================================================

print("\n" + "=" * 75)
print("🍇 2. STRATIFIED TRAIN / VAL / TEST SPLIT")
print("=" * 75)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train Set : {len(X_train)} images ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val Set   : {len(X_val)} images ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test Set  : {len(X_test)} images ({len(X_test)/len(X)*100:.1f}%)")

# ================================================================
# 5. DATA AUGMENTATION & MOBILENETV2 MODEL ARCHITECTURE
# ================================================================

print("\n" + "=" * 75)
print("🍇 3. BUILDING MOBILENETV2 MODEL")
print("=" * 75)

augmentation_layer = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15)
], name="grape_augmentation")

inputs = layers.Input(shape=(224, 224, 3))
x = augmentation_layer(inputs)

# MobileNetV2 expects input normalized to [-1, 1]
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = True

# Freeze initial 100 layers, fine-tune the rest
for layer in base_model.layers[:100]:
    layer.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs, name="Grape_MobileNetV2")
model.summary()

# ================================================================
# 6. COMPILE MODEL & CALLBACKS
# ================================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

cb_list = [
    callbacks.ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.CSVLogger(LOG_CSV_PATH)
]

# ================================================================
# 7. TRAINING EXECUTION
# ================================================================

print("\n" + "=" * 75)
print("🍇 4. STARTING MODEL TRAINING")
print("=" * 75)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=32,
    callbacks=cb_list,
    verbose=1
)

# ================================================================
# 8. TRAINING SUMMARY
# ================================================================

best_val_acc = max(history.history['val_accuracy'])
summary_dict = {
    'Plant': 'Grape',
    'Architecture': 'MobileNetV2 Transfer Learning',
    'Total_Samples': len(X),
    'Train_Samples': len(X_train),
    'Val_Samples': len(X_val),
    'Test_Samples': len(X_test),
    'Best_Val_Accuracy': float(best_val_acc),
    'Saved_Model_Path': BEST_MODEL_PATH
}

with open(SUMMARY_JSON, 'w') as f:
    json.dump(summary_dict, f, indent=2)

print("\n" + "=" * 75)
print("✅ GRAPE MODEL TRAINING COMPLETE")
print("=" * 75)
print(f"Best Validation Accuracy : {best_val_acc*100:.2f}%")
print(f"Model saved at           : {BEST_MODEL_PATH}")
print(f"Log saved at             : {LOG_CSV_PATH}")
print(f"Summary saved at         : {SUMMARY_JSON}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍇 1. LOADING GRAPE DATASET
Total samples : 1800
X shape       : (1800, 224, 224, 3) , dtype: uint8
y shape       : (1800,) , dtype: int64
Classes       : ['Healthy', 'Black_Rot', 'Leaf_Blight']

🍇 2. STRATIFIED TRAIN / VAL / TEST SPLIT
Train Set : 1260 images (70.0%)
Val Set   : 270 images (15.0%)
Test Set  : 270 images (15.0%)

🍇 3. BUILDING MOBILENETV2 MODEL
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "Grape_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ grape_augmentation (Sequential) │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 2,025,795 (7.73 MB)

 Non-trainable params: 396,544 (1.51 MB)


🍇 4. STARTING MODEL TRAINING
Epoch 1/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6298 - loss: 0.8483
Epoch 1: val_accuracy improved from None to 0.82222, saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/grape_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/grape_cnn_best.keras
40/40 ━━━━━━━━━━━━━━━━━━━━ 147s 3s/step - accuracy: 0.7579 - loss: 0.5819 - val_accuracy: 0.8222 - val_loss: 0.4099 - learning_rate: 1.0000e-04
Epoch 2/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9225 - loss: 0.2209
Epoch 2: val_accuracy did not improve from 0.82222
40/40 ━━━━━━━━━━━━━━━━━━━━ 128s 3s/step - accuracy: 0.9206 - loss: 0.2170 - val_accuracy: 0.8074 - val_loss: 0.4018 - learning_rate: 1.0000e-04
Epoch 3/25
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9457 - loss: 0.1415
Epoch 3: val_accuracy did not improve from 0.82222
40/40 ━━━━━━━━━━━━━